# Figure S1

Environment: `python_env` (`envs/python_env.yml`)

Input: `data/qPCR_MS2_strands.xlsx`

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
OUT_DIR = Path(config["out_dir"])
FIG_DIR = OUT_DIR / "figures"
SRC_DIR = OUT_DIR / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_excel(ROOT / "data" / "qPCR_MS2_strands.xlsx")[["Target", "Cq"]]
df["template"] = df.Target.str.split("-").str[0]
df["primers"] = df.Target.str.split("-").str[-1]

ref = (df[df.primers.str.upper() == "16S"].groupby("template").Cq.mean()
       .rename("ref_cq").reset_index())

df = df[df.primers.isin(["2P", "2I"]) & ~df.template.isin(["OB", "N", "NEG"])].merge(ref, on="template")
df["template"] = df.template.astype(float)
df["dcq"] = df.Cq - df.ref_cq

t0 = df[df.template == 0].groupby("primers").dcq.mean().rename("dcq_t0").reset_index()
df = df.merge(t0, on="primers")
df["ratio"] = 2 ** (df.dcq_t0 - df.dcq)
df["strand"] = df.primers.map({"2P": "Negative", "2I": "Positive"})

## Figure S1

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(data=df, x="template", hue="strand", hue_order=["Negative", "Positive"], y="ratio", errorbar="se",
            palette={"Positive": "#FF7F0E", "Negative": "#ffdd80"}, ax=ax)
ax.set_yscale("log")
ax.set_ylabel("Ratio number of genomes\nto time=0")
ax.set_xlabel("Time (minutes post infection)")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 0.7))
ax.grid(alpha=0.2)

fig.savefig(FIG_DIR / "FigS1.png", dpi=800, bbox_inches="tight")
fig.savefig(FIG_DIR / "FigS1.pdf", bbox_inches="tight")
plt.show()

df[["Target", "template", "strand", "Cq", "ref_cq", "ratio"]].to_csv(SRC_DIR / "FigS1_data.csv", index=False)